What is LSTM? 

LSTM: Long Short Term Memory is a neural network architecture that allows RNNs to retain information over long sequences. It has a gradient highway where the previous layer's output is passed through and added to the processed input token. The cell state allows information to persist. There are 3 gating mechanisms thea control the flow of information. A forget gate which decides whther to discard the information from the previous cell state. Input gate that determines which new information to store in the cell state. Output gate that decides what part of the cell state to output as the hidden state/, 


Gates that matter for volatility forecasting 
Forget gate is important as it allows us to just focus on the volatility numbers that matter such as sudden drops or sudden increases in prices. 
input gate is also important as it determines whether the current input is important

We need to feed into LSTM log_returns as we need continuous data about the exact price change of a feature. We also need to add volatility as it captures the recent turbulence and variability the market is in right now. Using both it can give us a context of how th emarket is right now. Raw prices would not work as othe values arent normalized.

The tradeoffs of using a longer window would be that we have more data about how the feature moves along time. Thus, we can determine if an abrupt change in price or high volatility is common or is uncommon. Using shorter windows allows us to determine the best case scenario at the moment like a greedy algorithm. I think the LSTm for bitcoin should lookback around 30 days. 

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import torch
import torch.nn as nn



In [2]:
def create_sequences(df: pd.DataFrame, window_size: int):
    X_data = df[['log_returns','volatility']].values
    y_data = df['realized_variance'].values
    X = [X_data[i:i+window_size] for i in range(len(df)-window_size)]
    y = [y_data[i+window_size] for i in range(len(df)-window_size)]
    
    return torch.tensor(np.array(X), dtype = torch.float32), torch.tensor(np.array(y), dtype = torch.float32)

In [3]:
# load csv 
btc_df = pd.read_csv('../data/btc_data.csv', index_col = 'Date', parse_dates = True)

train = btc_df.loc["2018-01-31":"2022-12-31"]
val = btc_df.loc["2023-01-01":"2023-12-31"]
test = btc_df.loc["2024-01-01":"2024-12-31"]

print(btc_df.index)

DatetimeIndex(['2018-01-31', '2018-02-01', '2018-02-02', '2018-02-03',
               '2018-02-04', '2018-02-05', '2018-02-06', '2018-02-07',
               '2018-02-08', '2018-02-09',
               ...
               '2024-12-21', '2024-12-22', '2024-12-23', '2024-12-24',
               '2024-12-25', '2024-12-26', '2024-12-27', '2024-12-28',
               '2024-12-29', '2024-12-30'],
              dtype='datetime64[us]', name='Date', length=2526, freq=None)


In [ ]:
# Scale train input data 
from sklearn.preprocessing import StandardScaler

log_scaler = StandaradScaler()
vol_scaler = StandardScaler()
train_log_return = log_scaler.fit_transform(train['log_returns'])
train_volatility = vol_scaler.fit_transform(train['volatility'])
val_log_return = log_scaler.fit(val['log_returns'])
test_log_return = log_scaler.fit(test['log_returns'])
val_volatility = log_scaler.fit(val['volatility'])
test_volatility = log_scaler.fit(test['volatility'])

#Scale realized_variance

scaled_train_variance = train['realized_variance'] * 10000
scaled_val_variance = val['realized_variance'] * 10000
scaled_test_variance = test['realized_variance'] * 10000

#Concat scaled values
train = pd.concat(train_log_return,train_volatility,scaled_train_variance)
val = pd.concat(val_log_return, val_volatility, scaled_val_variance)
test = pd.concat(test_log_return, test_volatility, scaled_test_variance)




In [4]:
#Creating train, val, and test sets 

X_train, y_train = create_sequences(train, window_size = 30)
X_test, y_test = create_sequences(test, window_size = 30)
X_val, y_val = create_sequences(val, window_size = 30)

print(f"X_train shape: {X_train.shape}\ny_train shape: {y_train.shape}")




X_train shape: torch.Size([1766, 30, 2])
y_train shape: torch.Size([1766])


In [5]:
import sys
import os

current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

if project_root not in sys.path:
    sys.path.append(project_root)
    
from models.lstm import LSTMModel


input_size = 2 # Two inputs volatility and log returns
hidden_size = 64 # Number of units in the hidden state
num_layers = 2 # Two stacked LSTMs
dropout = 0.2 # Dropout rate of 20%
learning_rate = 1e-3

#Creating LSTM model
model = LSTMModel(input_size = input_size, 
                       hidden_size = hidden_size,
                       num_layers = num_layers, 
                       dropout = dropout)

# MSE loss function 
loss_fn = nn.MSELoss()
# Adam Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

Importance of batch size

Small batch result in more noise into gradient updates which acts as a regularizer. Helps model escape local minima. 
Computting entire dataset gives you precise gradient but it might overfit. 

We shuffle the dataloader in training as each X[i] is a window_size period of data. We're shuffling the order of the specific windows. This is important as each avoids the model to learn patterns across windows. Itt also prevents gradient updates from being correlated as consecutive batches share similar markets which can bias the gradient direction. 

In [6]:
# Setting up dataloaders
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, shuffle = True, batch_size = 128)
test_loader = DataLoader(test_ds, batch_size = 128)
val_loader = DataLoader(val_ds, batch_size = 128)


In [7]:

if os.path.exists('../models/mse_lstm_model.pth'):
    model = LSTMModel(input_size=2, hidden_size=64, num_layers=2, dropout=0.2)
    model.load_state_dict(torch.load('../models/mse_lstm_model.pth'))
    model.eval()
    print("Loaded from checkpoint")
else:
    #Training loop 
    print("Training from scratch")
    model.train()

    num_epochs = 50

    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        #Train
        model.train()
        running_train_loss = 0
        for X_batch, y_batch in train_loader:
            
            X_batch, y_batch = X_batch.float(), y_batch.float()
            #Forward pass
            pred = model(X_batch)
            loss = loss_fn(pred, y_batch.view(-1,1))
            
            #Backprop and optimization 
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item()
            
        avg_train_loss = running_train_loss/len(train_loader)
        train_losses.append(avg_train_loss)
            
        #Validation
        model.eval()
        running_val_loss = 0
        with torch.no_grad(): # Used as no gradient computation is needed since there is no backprop on validation
            for X_b_val, y_b_val in val_loader:
                
                X_b_val, y_b_val = X_b_val.float(), y_b_val.float()
                
                #Forward pass
                val_pred = model(X_b_val)
                v_loss = loss_fn(val_pred, y_b_val.view(-1,1))
                
                running_val_loss += v_loss.item()
                
            avg_val_loss = running_val_loss/len(val_loader)
            val_losses.append(avg_val_loss)
        
        if (epoch + 1) % 10 == 0 or epoch == 0 or (epoch+1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] | "
                f"Train Loss: {avg_train_loss:.6f} | "
                f"Val Loss: {avg_val_loss:.6f}")
    #Save model
    torch.save(model.state_dict(), '../models/mse_lstm_model.pth')
        

Training from scratch
Epoch [1/50] | Train Loss: 0.320391 | Val Loss: 0.147793
Epoch [10/50] | Train Loss: 0.000157 | Val Loss: 0.000134
Epoch [20/50] | Train Loss: 0.000088 | Val Loss: 0.000067
Epoch [30/50] | Train Loss: 0.000069 | Val Loss: 0.000046
Epoch [40/50] | Train Loss: 0.000059 | Val Loss: 0.000035
Epoch [50/50] | Train Loss: 0.000052 | Val Loss: 0.000027


On normal mse loss Val loss is constantly lower than train loss probably due to the training data consisting of a larger period of time than the validation data. It is also lower because during validation dropout is not applied, meaning that the model is utilizing all its neurons for predictiona

In [8]:
# Running model on test set 
model.eval()
running_test_loss = 0 
test_losses = []
predictions = []

with torch.no_grad():
    for X_btest, y_btest in test_loader:
        X_btest, y_btest = X_btest.float(), y_btest.float()
        
        #Forward pass
        val_pred = model(X_btest)
        predictions.append(val_pred)
        v_loss = loss_fn(val_pred, y_btest.view(-1,1))
        
        running_test_loss += v_loss.item()
        
    avg_test_loss = running_test_loss/len(test_loader)
    test_losses.append(avg_test_loss)
    
print(predictions)

[tensor([[0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [0.0056],
        [

In [9]:
#Create Dataframe 
window_size = 30

actual = test['volatility'].iloc[window_size:]
forecast_index = test.index[window_size:]
lstm_forecast = torch.cat(predictions).cpu().numpy().flatten()

results = pd.DataFrame({'actual': actual, 'lstm_forecast':lstm_forecast}, index= forecast_index)

print(results.head)

<bound method NDFrame.head of               actual  lstm_forecast
Date                               
2024-01-31  0.028174       0.005587
2024-02-01  0.028052       0.005587
2024-02-02  0.026637       0.005587
2024-02-03  0.026022       0.005587
2024-02-04  0.026069       0.005587
...              ...            ...
2024-12-26  0.024819       0.005587
2024-12-27  0.023788       0.005587
2024-12-28  0.023869       0.005587
2024-12-29  0.023791       0.005587
2024-12-30  0.023780       0.005587

[335 rows x 2 columns]>


In [10]:
# MSE MAE and Directional accuracy comparisons
#Computing MSE and MAE 
from sklearn.metrics import mean_absolute_error, mean_squared_error

mse = mean_squared_error(results['actual'], results['lstm_forecast'])
mae = mean_absolute_error(results['actual'], results['lstm_forecast'])

print(f"MSE: {mse}")
print(f"MAE: {mae}")

#Directional Accuracy

num_dates = len(results)
actual_diff = np.sign(results['actual'].diff().dropna())
pred_diff = np.sign(results['lstm_forecast'].diff().dropna())
correct_preds = actual_diff == pred_diff
directional_accuracy = (correct_preds).mean()
print(f"Correct predictions: {sum(correct_preds)}")
print(f"Total days: {num_dates}")
print(f"Directional Accuracy: {directional_accuracy *100}")
    
    

MSE: 0.00050166174365755
MAE: 0.021398556318842143
Correct predictions: 113
Total days: 335
Directional Accuracy: 33.83233532934132


Lower MSE and MAE compared to garch but significantly lower directional accuracy (69.5% vs 50.8%)

We trained the model to minimize MSE loss as it was our loss function. As a result, the precision of the model incresaed heavily meaning tht the forecasted values where very close to the actual values. However, tis reduced variability in the predictions menaing that most predictions were close to the mea, heence overfit to the data. 

Additionally, the model might have learned to simply predict today's volatility as tommorowws's volatility (naive lag-1 forecast). As volatility changes slowly, it would result in low MSE if the model just copies the previous day's volatility. 

We can compare the correlations between lstm_predictions and the actual results shifted by a day


In [65]:
correlation = results['lstm_forecast'].corr(results['actual'].shift(1))

print(f"Correlation between lstm prediction and yesterday's day volatilities: {correlation}")

Correlation between lstm prediction and yesterday's day volatilities: 0.9739265144694281


There is high correlation between LSTM predictions and yesterday's actual volatility suggesting that the model learned to copy yesterday's value rather than forecasting.

We could change the error to penalize on incorrect directionality.

total_loss = MSE_loss + lambda * directional_penalty


In [66]:
def directional_mse_loss(pred, actual, actual_yesterday, lam = 0.1):
    
    #Utilizing tanh to approximate sign and have valid gradient
    pred_diff = torch.tanh((pred - actual_yesterday) * 10 )
    actual_diff = torch.tanh((actual - actual_yesterday) * 10)
    
    mse = nn.MSELoss()(pred, actual)
    dir_penalty = (-pred_diff * actual_diff).mean() 
    # Set negative sign to flip the outputs. This way same direction = -1 which minimizes loss, and diff direction = 1.
    
    return mse + lam * dir_penalty
    

In [ ]:
# Update training loop with directional_mse_loss
import torch.nn.functional as F

if os.path.exists('../models/directional_mse_lstm_model.pth'):
    model = LSTMModel(input_size=2, hidden_size=64, num_layers=2, dropout=0.2)
    model.load_state_dict(torch.load('../models/directional_mse_lstm_model.pth'))
    model.eval()
    print("Loaded from checkpoint")
else:
    model = LSTMModel(input_size=2, hidden_size=64, num_layers=2, dropout=0.2)
    optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)
    num_epochs = 50

    train_losses = []
    val_losses = []
    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0
        for X_train_batch, y_train_batch in train_loader:
            
            # Convert input and output into 32 bit floating integers
            X_train_batch, y_train_batch = X_train_batch.float(), y_train_batch.float()
            
            # Take the day 30 volatility from all training batches
            actual_yesterday = X_train_batch[:, -1, 1].view(-1,1)
            
            # Forward pass
            pred = model(X_train_batch) 
            # Calculates loss based on predictions
            loss = directional_mse_loss(pred, y_train_batch.view(-1,1), actual_yesterday)
            
            # Backpropagation
            optimizer.zero_grad() # Clears out old gradients from previous batch 
            loss.backward() # Applies backprop
            optimizer.step() # Optimizer tweaks model's weights
            
            running_train_loss += loss.item() # Extracts raw python float values from loss tensor and accumulate 
                
        avg_train_loss = running_train_loss/len(train_loader)
        train_losses.append(avg_train_loss) #Append batch avg_training loss

        #Validation
        model.eval()
        running_val_loss = 0
        
        with torch.no_grad(): # Makes pytorch not track gradients
            for X_b_val, y_b_val in val_loader:
                
                X_b_val, y_b_val = X_b_val.float(), y_b_val.float().view(-1,1)
                
                actual_yesterday = X_b_val[:,-1,1].view(-1,1)
                
                #Forward pass
                val_pred = model(X_b_val)
                v_loss = directional_mse_loss(val_pred, y_b_val, actual_yesterday)
                
                running_val_loss += v_loss.item()
                
            avg_val_loss = running_val_loss/len(val_loader)
            val_losses.append(avg_val_loss)
        
        if (epoch + 1) % 10 == 0 or epoch == 0 or (epoch+1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] | "
                f"Train Loss: {avg_train_loss:.6f} | "
                f"Val Loss: {avg_val_loss:.6f}")
    #Save model
    torch.save(model.state_dict(), '../models/directional_mse_lstm_model.pth')
            
            
            
            
            

Epoch [1/50] | Train Loss: 0.005496 | Val Loss: 0.001721
Epoch [10/50] | Train Loss: 0.000160 | Val Loss: 0.000197
Epoch [20/50] | Train Loss: 0.000155 | Val Loss: 0.000184
Epoch [30/50] | Train Loss: 0.000117 | Val Loss: 0.000194
Epoch [40/50] | Train Loss: 0.000060 | Val Loss: 0.000131
Epoch [50/50] | Train Loss: 0.000041 | Val Loss: 0.000036


In [70]:
# Running model on test set 
model.eval()
running_test_loss = 0 
test_losses = []
predictions = []

with torch.no_grad():
    for X_btest, y_btest in test_loader:
        X_btest, y_btest = X_btest.float(), y_btest.float()
        
        #Forward pass
        val_pred = model(X_btest)
        predictions.append(val_pred)
        v_loss = loss_fn(val_pred, y_btest.view(-1,1))
        
        running_test_loss += v_loss.item()
        
    avg_test_loss = running_test_loss/len(test_loader)
    test_losses.append(avg_test_loss)
    
print(predictions)

[tensor([[0.0311],
        [0.0313],
        [0.0313],
        [0.0312],
        [0.0311],
        [0.0311],
        [0.0310],
        [0.0308],
        [0.0300],
        [0.0292],
        [0.0282],
        [0.0276],
        [0.0270],
        [0.0261],
        [0.0256],
        [0.0249],
        [0.0247],
        [0.0247],
        [0.0249],
        [0.0250],
        [0.0252],
        [0.0252],
        [0.0252],
        [0.0252],
        [0.0253],
        [0.0250],
        [0.0246],
        [0.0237],
        [0.0228],
        [0.0217],
        [0.0222],
        [0.0228],
        [0.0238],
        [0.0247],
        [0.0247],
        [0.0266],
        [0.0278],
        [0.0289],
        [0.0297],
        [0.0305],
        [0.0311],
        [0.0310],
        [0.0314],
        [0.0315],
        [0.0322],
        [0.0331],
        [0.0348],
        [0.0353],
        [0.0360],
        [0.0380],
        [0.0382],
        [0.0394],
        [0.0407],
        [0.0416],
        [0.0416],
        [

In [71]:
#Create Dataframe 
window_size = 30

actual = test['volatility'].iloc[window_size:]
forecast_index = test.index[window_size:]
lstm_forecast = torch.cat(predictions).cpu().numpy().flatten()

results = pd.DataFrame({'actual': actual, 'lstm_forecast':lstm_forecast}, index= forecast_index)

print(results.head)

<bound method NDFrame.head of               actual  lstm_forecast
Date                               
2024-01-31  0.028174       0.031110
2024-02-01  0.028052       0.031305
2024-02-02  0.026637       0.031287
2024-02-03  0.026022       0.031185
2024-02-04  0.026069       0.031095
...              ...            ...
2024-12-26  0.024819       0.029954
2024-12-27  0.023788       0.030122
2024-12-28  0.023869       0.030292
2024-12-29  0.023791       0.030105
2024-12-30  0.023780       0.030136

[335 rows x 2 columns]>


In [72]:
# MSE MAE and Directional accuracy comparisons
#Computing MSE and MAE 
from sklearn.metrics import mean_absolute_error, mean_squared_error

mse = mean_squared_error(results['actual'], results['lstm_forecast'])
mae = mean_absolute_error(results['actual'], results['lstm_forecast'])

print(f"MSE: {mse}")
print(f"MAE: {mae}")

#Directional Accuracy

num_dates = len(results)
actual_diff = np.sign(results['actual'].diff().dropna())
pred_diff = np.sign(results['lstm_forecast'].diff().dropna())
correct_preds = actual_diff == pred_diff
directional_accuracy = (correct_preds).mean()
print(f"Correct predictions: {sum(correct_preds)}")
print(f"Total days: {num_dates}")
print(f"Directional Accuracy: {directional_accuracy *100}")
    
    

MSE: 2.6634078139850448e-05
MAE: 0.0043834901904363675
Correct predictions: 175
Total days: 335
Directional Accuracy: 52.39520958083832


With the new loss function evaluating on directional movement resulted in a small increase in directional accuracy and significantly worse MSE and MAE losses than just using MSE loss. This is most likely due to the fact that 30-day rolling volatility is very smooth, and as we're predicting based on the average of th elast day, directionaly caccuracy can result as a very noisy metric. 

In an attempt to evaluate a smoother evaluation metric, qlike loss will be tested on the model. 

In [73]:
def qlike_loss(actual, pred, epsilon = 1e-10):
    actual = torch.clip(actual, epsilon, None)
    pred = torch.clip(pred, epsilon, None)
    
    loss = (actual/pred) - torch.log(actual/pred) -1
    
    return loss.mean()

In [90]:
# Test softplus is working
model = LSTMModel(input_size=2, hidden_size=64, num_layers=2, dropout=0.2)
X_sample, y_sample = next(iter(train_loader))
pred = model(X_sample.float())
print(f"pred min: {pred.min().item()}, pred max: {pred.max().item()}")

pred min: 0.03664763271808624, pred max: 0.04886310175061226


In [91]:
# Train model on qlike loss 
if os.path.exists('../models/qlike_lstm_model.pth'):
    model = LSTMModel(input_size=2, hidden_size=64, num_layers=2, dropout=0.2)
    model.load_state_dict(torch.load('../models/qlike_lstm_model.pth'))
    model.eval()
    print("Loaded from checkpoint")
else:
    model = LSTMModel(input_size=2, hidden_size=64, num_layers=2, dropout=0.2)
    optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)
    
    num_epochs = 50
    
    test_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        model.train()
        running_train_loss =0
        

        for X_train_batch, y_train_batch in train_loader:
            # Forward
            pred = model(X_train_batch)
            
            
            loss = qlike_loss(y_train_batch, pred)
            
            # Backprop
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item()
        
        avg_train_loss = running_train_loss/len(train_loader)
        train_losses.append(avg_train_loss) #Append batch avg_training loss
        
        model.eval()
        running_val_loss = 0
        with torch.no_grad():
            for X_b_val, y_b_val in val_loader:
                X_b_val, y_b_val = X_b_val.float(), y_b_val.float().view(-1,1)
                #Forward pass
                val_pred = model(X_b_val)        
                squared_y = torch.square(y_b_val)
                square_pred = torch.square(val_pred)
                v_loss = qlike_loss(squared_y, square_pred)
                
                running_val_loss += v_loss.item()
                
            avg_val_loss = running_val_loss/len(val_loader)
            val_losses.append(avg_val_loss)
             
             
        if (epoch + 1) % 10 == 0 or epoch == 0 or (epoch+1) % 50 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}] | "
                    f"Train Loss: {avg_train_loss:.6f} | "
                    f"Val Loss: {avg_val_loss:.6f}")
    #Save model
    torch.save(model.state_dict(), '../models/qlike_lstm_model.pth')
            
         

Epoch [1/50] | Train Loss: 0.100033 | Val Loss: 0.361719
Epoch [10/50] | Train Loss: 0.072410 | Val Loss: 0.377804
Epoch [20/50] | Train Loss: 0.068189 | Val Loss: 0.427688
Epoch [30/50] | Train Loss: 0.069732 | Val Loss: 0.345494
Epoch [40/50] | Train Loss: 0.068800 | Val Loss: 0.419594
Epoch [50/50] | Train Loss: 0.069066 | Val Loss: 0.501681


In [93]:
# Running model on test set 
model.eval()
running_test_loss = 0 
test_losses = []
predictions = []

with torch.no_grad():
    for X_btest, y_btest in test_loader:
        X_btest, y_btest = X_btest.float(), y_btest.float()
        
        #Forward pass
        val_pred = model(X_btest)
        predictions.append(val_pred)
        v_loss = qlike_loss(val_pred, y_btest.view(-1,1))
        
        running_test_loss += v_loss.item()
        
    avg_test_loss = running_test_loss/len(test_loader)
    test_losses.append(avg_test_loss)
    
print(predictions)

[tensor([[0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0377],
        [0.0377],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [0.0378],
        [